# UIT DSC 2026 LegalIR - Step 5 Fine-tune Vietnamese Bi-Encoder

Fine-tune `bkai-foundation-models/vietnamese-bi-encoder` with `MultipleNegativesRankingLoss`, retrieve dense candidates, fuse them with Step 4 RRF candidates, and create `submission.zip`.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'sentence-transformers', 'accelerate'], check=True)


## Config And Input Paths

`step4.zip` dataset already contains `chunks.jsonl`, `train_split.json`, and `dev_split.json`. `step5.zip` only needs to add Step 4 RRF outputs: `dev_rankings_rrf.jsonl`, `public_rankings_rrf.jsonl`, plus small provenance/config files.

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import time
import zipfile
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np

# Avoid SentenceTransformers/Trainer DataParallel device-side asserts on Kaggle T4x2.
# Step 5 baseline trains on one T4; later DDP can be added if we need more throughput.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
OUTPUT_DIR = Path('/kaggle/working/step5')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')
MODEL_NAME = 'bkai-foundation-models/vietnamese-bi-encoder'
ALLOWED_MODELS = {
    'BAAI/bge-m3',
    'bkai-foundation-models/vietnamese-bi-encoder',
    'itdainb/PhoRanker',
    'Qwen/Qwen3-Reranker-0.6B',
}
MAX_SUBMISSION_DOCS = 5

# Path aliases only handle Kaggle folder nesting or renamed copies of the same
# Step 4 artifacts. They must not switch to another step's candidate pool.
def require_existing(name: str, candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Missing required Step 5 input: {name}. Checked:\n{checked}')

STEP4_ROOTS = [DATA_ROOT / 'step4', DATA_ROOT / 'step4' / 'step4']
STEP5_INPUT_ROOTS = [DATA_ROOT / 'step5', DATA_ROOT / 'step5' / 'step5']
CHUNKS_FILE = require_existing('step4/chunks.jsonl', [root / 'chunks.jsonl' for root in STEP4_ROOTS])
TRAIN_FILE = require_existing('step4/train_split.json', [root / 'train_split.json' for root in STEP4_ROOTS])
DEV_FILE = require_existing('step4/dev_split.json', [root / 'dev_split.json' for root in STEP4_ROOTS])
STEP4_BEST_CONFIG = require_existing('step4/best_config.json', [root / 'best_config.json' for root in STEP4_ROOTS])
STEP4_CONFIG_FILE = require_existing(
    'step4 config artifact',
    [root / 'step4_config.json' for root in STEP5_INPUT_ROOTS] + [root / 'best_config.json' for root in STEP4_ROOTS],
)
STEP4_TRAIN_RANKINGS = require_existing('step4/train_rankings_best.jsonl', [root / 'train_rankings_best.jsonl' for root in STEP4_ROOTS])
STEP4_DEV_RANKINGS = require_existing(
    'step4/dev rankings',
    [root / 'dev_rankings_rrf.jsonl' for root in STEP5_INPUT_ROOTS]
    + [root / 'dev_rankings_best.jsonl' for root in STEP4_ROOTS]
    + [root / 'rankings' / 'dev_rankings_rrf.jsonl' for root in STEP4_ROOTS],
)
STEP4_PUBLIC_RANKINGS = require_existing(
    'step4/public rankings',
    [root / 'public_rankings_rrf.jsonl' for root in STEP5_INPUT_ROOTS]
    + [root / 'public_rankings_best.jsonl' for root in STEP4_ROOTS]
    + [root / 'rankings' / 'public_rankings_rrf.jsonl' for root in STEP4_ROOTS],
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)
print('STEP4_ROOTS:', STEP4_ROOTS)
print('STEP5_INPUT_ROOTS:', STEP5_INPUT_ROOTS)
print('CHUNKS_FILE:', CHUNKS_FILE)
print('TRAIN_FILE:', TRAIN_FILE)
print('DEV_FILE:', DEV_FILE)
print('STEP4_BEST_CONFIG:', STEP4_BEST_CONFIG)
print('STEP4_CONFIG_FILE:', STEP4_CONFIG_FILE)
print('STEP4_TRAIN_RANKINGS:', STEP4_TRAIN_RANKINGS)
print('STEP4_DEV_RANKINGS:', STEP4_DEV_RANKINGS)
print('STEP4_PUBLIC_RANKINGS:', STEP4_PUBLIC_RANKINGS)
print('OUTPUT_DIR:', OUTPUT_DIR)


## Utilities

In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, sort_keys=True)
        f.write('\n')

def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')

def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', text).replace('?', 'd').replace('?', 'D')

TOKEN_RE = re.compile(r'\b\w+\b', flags=re.UNICODE)
STOPWORDS = {
    'a','an','anh','ay','bi','boi','cac','can','cho','co','con','cua','duoc','da','de','den','di','do','doi','duoi','gi','hay','hoac','khi','la','lai','lam','mot','nay','neu','nhu','nhung','o','phai','qua','quy','rieng','sau','se','thi','theo','thuoc','toi','trong','tu','va','ve','vi','voi'
}

def tokenize(text: str) -> list[str]:
    text = strip_accents(text.lower())
    return [tok for tok in TOKEN_RE.findall(text) if len(tok) > 1 and tok not in STOPWORDS]

def evaluate_rankings(rankings: dict[str, list[str]], payload: dict[str, Any]) -> dict[str, Any]:
    rows = []
    for qid, row in payload.items():
        gold = [str(x) for x in row.get('answer', [])] if isinstance(row, dict) else []
        pred = [str(x) for x in rankings.get(str(qid), [])]
        top5 = pred[:MAX_SUBMISSION_DOCS]
        gold_set = set(gold)
        denom = len(gold_set) if gold_set else 1
        top5_hits = sum(1 for doc_id in top5 if doc_id in gold_set)
        hit_positions = [idx + 1 for idx, doc_id in enumerate(pred) if doc_id in gold_set]
        rows.append({
            'query_id': str(qid),
            'num_gold': len(gold_set),
            'exist@90': 1.0 if any(doc_id in gold_set for doc_id in pred[:90]) else 0.0,
            'hit@1': 1.0 if pred[:1] and pred[0] in gold_set else 0.0,
            'hit@5': 1.0 if top5_hits else 0.0,
            'hit@20': 1.0 if any(doc_id in gold_set for doc_id in pred[:20]) else 0.0,
            'mrr': 1.0 / hit_positions[0] if hit_positions else 0.0,
            'precision@5': top5_hits / MAX_SUBMISSION_DOCS,
            'recall@1': sum(1 for doc_id in pred[:1] if doc_id in gold_set) / denom,
            'recall@5': top5_hits / denom,
            'recall@20': sum(1 for doc_id in pred[:20] if doc_id in gold_set) / denom,
            'recall@50': sum(1 for doc_id in pred[:50] if doc_id in gold_set) / denom,
            'recall@90': sum(1 for doc_id in pred[:90] if doc_id in gold_set) / denom,
            'recall@100': sum(1 for doc_id in pred[:100] if doc_id in gold_set) / denom,
        })
    keys = [k for k in rows[0] if k not in {'query_id', 'num_gold'}] if rows else []
    return {'macro': {key: float(np.mean([row[key] for row in rows])) for key in keys}, 'per_query': rows}

def make_submission(predictions: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    return {str(qid): {'answer': [str(doc_id) for doc_id in docs[:MAX_SUBMISSION_DOCS]]} for qid, docs in predictions.items()}

def validate_submission_payload(submission: dict[str, Any], public_payload: dict[str, Any], valid_doc_ids: set[str]) -> dict[str, Any]:
    issues = []
    expected_qids = set(map(str, public_payload.keys()))
    actual_qids = set(map(str, submission.keys()))
    for qid in sorted(expected_qids - actual_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'missing query_id'})
    for qid in sorted(actual_qids - expected_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'unexpected query_id'})
    lengths = Counter()
    for qid, row in submission.items():
        answer = row.get('answer') if isinstance(row, dict) else None
        if not isinstance(answer, list):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer must be a list'})
            continue
        lengths[str(len(answer))] += 1
        if not (1 <= len(answer) <= MAX_SUBMISSION_DOCS):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer length must be 1..5'})
        if len(answer) != len(set(answer)):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'duplicate document_id'})
        for doc_id in answer:
            if not isinstance(doc_id, str):
                issues.append({'level': 'error', 'query_id': str(qid), 'message': 'document_id must be string'})
            elif doc_id not in valid_doc_ids:
                issues.append({'level': 'error', 'query_id': str(qid), 'message': f'unknown document_id: {doc_id}'})
    return {
        'num_public_queries': len(expected_qids),
        'num_submission_queries': len(actual_qids),
        'answer_length_distribution': dict(sorted(lengths.items())),
        'num_errors': sum(1 for issue in issues if issue['level'] == 'error'),
        'num_warnings': sum(1 for issue in issues if issue['level'] == 'warning'),
        'issues': issues[:200],
    }

def write_submission_zip(submission_json: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_json, arcname='submission.json')


## Load Corpus And Build Positive Pairs

In [ ]:
def load_chunks(chunks_file: Path) -> tuple[list[dict[str, Any]], dict[str, list[int]], set[str]]:
    chunks = []
    doc_to_chunk_indices: dict[str, list[int]] = defaultdict(list)
    valid_doc_ids = set()
    for idx, row in enumerate(iter_jsonl(chunks_file)):
        doc_id = str(row.get('doc_id', ''))
        chunk = {
            'chunk_idx': idx,
            'chunk_id': str(row.get('chunk_id', idx)),
            'doc_id': doc_id,
            'text': str(row.get('text', '')),
            'heading': str(row.get('heading', '')),
            'word_count': int(row.get('word_count') or 0),
            'metadata': row.get('metadata') if isinstance(row.get('metadata'), dict) else {},
        }
        chunks.append(chunk)
        if doc_id:
            doc_to_chunk_indices[doc_id].append(idx)
            valid_doc_ids.add(doc_id)
        if (idx + 1) % 50000 == 0:
            print(f'loaded {idx + 1:,} chunks')
    return chunks, doc_to_chunk_indices, valid_doc_ids

def select_positive_chunks(question: str, gold_doc_id: str, chunks: list[dict[str, Any]], doc_to_chunk_indices: dict[str, list[int]], *, top_n: int) -> list[dict[str, Any]]:
    q_tokens = Counter(tokenize(question))
    q_set = set(q_tokens)
    q_deaccent = strip_accents(question.lower())
    scored = []
    for chunk_idx in doc_to_chunk_indices.get(str(gold_doc_id), []):
        chunk = chunks[chunk_idx]
        c_tokens = tokenize(chunk['text'][:6000])
        c_counter = Counter(c_tokens)
        overlap = sum(min(q_tokens[tok], c_counter[tok]) for tok in q_set)
        heading = strip_accents(chunk['heading'].lower())
        heading_bonus = 0.25 if any(tok in heading for tok in q_set) else 0.0
        exact_bonus = 0.5 if q_deaccent[:80] and q_deaccent[:80] in strip_accents(chunk['text'].lower()) else 0.0
        score = overlap / max(1, len(q_set)) + heading_bonus + exact_bonus
        scored.append((score, -abs(chunk['word_count'] - 320), chunk_idx))
    scored.sort(reverse=True)
    return [chunks[chunk_idx] for _, _, chunk_idx in scored[:top_n] if chunks[chunk_idx]['text'].strip()]

def build_train_pairs(train_payload: dict[str, Any], *, positives_per_gold: int, max_examples: int) -> list[dict[str, Any]]:
    pairs = []
    missing_gold_docs = 0
    for qid, row in train_payload.items():
        question = row.get('question', '') if isinstance(row, dict) else ''
        gold_docs = [str(x) for x in row.get('answer', [])] if isinstance(row, dict) else []
        for doc_id in gold_docs:
            positives = select_positive_chunks(question, doc_id, chunks, doc_to_chunk_indices, top_n=positives_per_gold)
            if not positives:
                missing_gold_docs += 1
                continue
            for chunk in positives:
                pairs.append({
                    'query_id': str(qid),
                    'doc_id': doc_id,
                    'chunk_id': chunk['chunk_id'],
                    'question': question,
                    'positive_text': chunk['text'],
                    'positive_heading': chunk['heading'],
                })
    random.shuffle(pairs)
    if max_examples and len(pairs) > max_examples:
        pairs = pairs[:max_examples]
    report = {'num_pairs': len(pairs), 'missing_gold_docs': missing_gold_docs, 'positives_per_gold': positives_per_gold, 'max_examples': max_examples}
    write_json(OUTPUT_DIR / 'training' / 'train_pair_report.json', report)
    append_jsonl(OUTPUT_DIR / 'training' / 'train_pairs.jsonl', pairs)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return pairs

train_payload = read_json(TRAIN_FILE)
dev_payload = read_json(DEV_FILE)
chunks, doc_to_chunk_indices, valid_doc_ids = load_chunks(CHUNKS_FILE)
print(f'chunks={len(chunks):,}; docs={len(valid_doc_ids):,}; train_queries={len(train_payload):,}; dev_queries={len(dev_payload):,}')


## Fine-tune With MultipleNegativesRankingLoss

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

@dataclass(frozen=True)
class Step5Config:
    model_name: str = MODEL_NAME
    positives_per_gold: int = 2
    max_train_examples: int = 16000
    train_batch_size: int = 12
    epochs: int = 1
    learning_rate: float = 4e-5
    weight_decay: float = 0.02
    warmup_ratio: float = 0.1
    max_seq_length: int = 256
    encode_batch_size: int = 96
    query_batch_size: int = 128
    dense_top_chunks: int = 300
    dense_top_docs: int = 100
    evidence_per_doc: int = 3
    aggregate_mean_top3_weight: float = 0.20
    aggregate_support_weight: float = 0.05
    search_block_size: int = 32768
    rrf_k_values: tuple[int, ...] = (20, 40, 60, 80)
    step4_weights: tuple[float, ...] = (0.8, 1.0, 1.2)
    ft_dense_weights: tuple[float, ...] = (0.4, 0.7, 1.0, 1.3)

config = Step5Config()
write_json(OUTPUT_DIR / 'configs' / 'step5_config.json', asdict(config))

pairs = build_train_pairs(train_payload, positives_per_gold=config.positives_per_gold, max_examples=config.max_train_examples)
train_examples = [InputExample(texts=[row['question'], row['positive_text']]) for row in pairs]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=config.train_batch_size, drop_last=True)

model = SentenceTransformer(config.model_name, trust_remote_code=True)
transformer = model._first_module()
max_positions = getattr(getattr(transformer, 'auto_model', None).config, 'max_position_embeddings', None)
if max_positions:
    # RoBERTa/PhoBERT-style models reserve positions for special tokens.
    safe_max_seq_length = min(config.max_seq_length, max(32, int(max_positions) - 2))
else:
    safe_max_seq_length = config.max_seq_length
model.max_seq_length = safe_max_seq_length
print({'requested_max_seq_length': config.max_seq_length, 'model_max_position_embeddings': max_positions, 'active_max_seq_length': model.max_seq_length})
param_count = int(sum(p.numel() for p in model.parameters()))
if config.model_name not in ALLOWED_MODELS:
    raise ValueError(f'Model not whitelisted: {config.model_name}')
manifest = {
    'models': [
        {
            'model_id': 'BAAI/bge-m3',
            'role': 'step4_dense_zero_shot_candidates_used_for_fusion',
            'parameter_count': 568_000_000,
            'parameter_count_source': 'Step 4 approximate audit',
        },
        {
            'model_id': config.model_name,
            'role': 'fine_tuned_dense_biencoder',
            'parameter_count': param_count,
            'parameter_count_source': 'sum(p.numel() for p in SentenceTransformer parameters)',
        },
    ],
    'allowed_models': sorted(ALLOWED_MODELS),
    'total_known_parameters': 568_000_000 + param_count,
    'max_total_parameters': 4_000_000_000,
    'no_hosted_inference_or_api': True,
    'local_training_and_inference': True,
}
if manifest['total_known_parameters'] >= manifest['max_total_parameters']:
    raise ValueError('Parameter audit failed: total >= 4B')
write_json(OUTPUT_DIR / 'reports' / 'model_manifest.json', manifest)
write_json(OUTPUT_DIR / 'configs' / 'runtime_training_config.json', {**asdict(config), 'active_max_seq_length': model.max_seq_length, 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES')})
print(json.dumps({'model': config.model_name, 'params': param_count, 'train_batches': len(train_loader), 'active_max_seq_length': model.max_seq_length}, ensure_ascii=False, indent=2))

train_loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = math.ceil(len(train_loader) * config.epochs * config.warmup_ratio)
model_output_path = OUTPUT_DIR / 'models' / 'vietnamese-bi-encoder-mnrl'
model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=config.epochs,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': config.learning_rate},
    weight_decay=config.weight_decay,
    output_path=str(model_output_path),
    save_best_model=False,
    use_amp=True,
    show_progress_bar=True,
)
model.save(str(model_output_path))
print('Saved model:', model_output_path)


## Retrieve With Fine-tuned Dense Model

In [ ]:
def encode_chunks_to_memmap(model: Any, chunks: list[dict[str, Any]], output_path: Path, *, batch_size: int) -> np.ndarray:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path = output_path.with_suffix('.meta.json')
    if output_path.exists() and meta_path.exists():
        print('loading cached chunk embeddings:', output_path)
        return np.load(output_path, mmap_mode='r')
    probe = model.encode([chunks[0]['text']], batch_size=1, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    dim = int(probe.shape[1])
    arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float16, shape=(len(chunks), dim))
    started = time.time()
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        vecs = model.encode([row['text'] for row in batch], batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        arr[start:start + len(batch)] = vecs.astype(np.float16)
        if (start // batch_size + 1) % 50 == 0:
            print(f'encoded {min(start + batch_size, len(chunks)):,}/{len(chunks):,} chunks')
    arr.flush()
    write_json(meta_path, {'model_name': config.model_name, 'num_chunks': len(chunks), 'dim': dim, 'dtype': 'float16', 'seconds': round(time.time() - started, 3)})
    return np.load(output_path, mmap_mode='r')

def dense_search_torch(chunk_embeddings: np.ndarray, query_embeddings: np.ndarray, *, top_k: int, block_size: int) -> list[list[tuple[int, float]]]:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    query_tensor = torch.tensor(query_embeddings, dtype=torch.float16 if device == 'cuda' else torch.float32, device=device)
    results = []
    for q_idx in range(query_tensor.shape[0]):
        top_scores = None
        top_indices = None
        q = query_tensor[q_idx:q_idx + 1].T
        for start in range(0, chunk_embeddings.shape[0], block_size):
            block_np = np.asarray(chunk_embeddings[start:start + block_size], dtype=np.float16 if device == 'cuda' else np.float32)
            block = torch.tensor(block_np, device=device)
            scores = (block @ q).squeeze(1)
            vals, idxs = torch.topk(scores, k=min(top_k, scores.numel()))
            idxs = idxs + start
            if top_scores is None:
                top_scores, top_indices = vals, idxs
            else:
                merged_scores = torch.cat([top_scores, vals])
                merged_indices = torch.cat([top_indices, idxs])
                top_scores, order = torch.topk(merged_scores, k=min(top_k, merged_scores.numel()))
                top_indices = merged_indices[order]
        results.append([(int(idx), float(score)) for idx, score in zip(top_indices.detach().cpu().tolist(), top_scores.detach().cpu().tolist())])
    return results

def aggregate_dense_docs(chunk_hits: list[tuple[int, float]]) -> list[dict[str, Any]]:
    per_doc = defaultdict(list)
    for chunk_idx, score in chunk_hits[:config.dense_top_chunks]:
        doc_id = chunks[chunk_idx]['doc_id']
        if doc_id:
            per_doc[doc_id].append((score, chunk_idx))
    rows = []
    for doc_id, scored in per_doc.items():
        scored.sort(reverse=True)
        scores = [s for s, _ in scored]
        max_score = scores[0]
        mean_top3 = sum(scores[:3]) / min(3, len(scores))
        support_count = len(scored)
        doc_score = max_score + config.aggregate_mean_top3_weight * mean_top3 + config.aggregate_support_weight * support_count
        evidence = []
        for score, chunk_idx in scored[:config.evidence_per_doc]:
            row = chunks[chunk_idx]
            evidence.append({'chunk_id': row['chunk_id'], 'score': score, 'heading': row['heading'], 'word_count': row['word_count']})
        rows.append({'doc_id': doc_id, 'score': doc_score, 'max_chunk_score': max_score, 'mean_top3_chunk_score': mean_top3, 'support_count': support_count, 'evidence': evidence})
    rows.sort(key=lambda row: row['score'], reverse=True)
    return rows[:config.dense_top_docs]

def load_step4_rankings(path: Path) -> dict[str, list[str]]:
    rankings = {}
    for row in iter_jsonl(path):
        if 'fused_doc_ids' in row:
            rankings[str(row['query_id'])] = [str(x) for x in row['fused_doc_ids']]
        elif 'top_docs' in row:
            rankings[str(row['query_id'])] = [str(x['doc_id']) for x in row['top_docs']]
    return rankings

def rrf_fuse(branches: list[tuple[list[str], float]], *, rrf_k: int, top_docs: int = 100) -> list[str]:
    scores = defaultdict(float)
    first_seen = {}
    for doc_ids, weight in branches:
        for rank, doc_id in enumerate(doc_ids, start=1):
            scores[doc_id] += weight / (rrf_k + rank)
            first_seen.setdefault(doc_id, len(first_seen))
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: (-item[1], first_seen[item[0]]))[:top_docs]]

def dense_rank_payload(payload: dict[str, Any], output_dense_file: Path) -> dict[str, list[str]]:
    query_items = list(payload.items())
    questions = [row.get('question', '') if isinstance(row, dict) else '' for _, row in query_items]
    query_embeddings = model.encode(questions, batch_size=config.query_batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    hits = dense_search_torch(chunk_embeddings, query_embeddings, top_k=config.dense_top_chunks, block_size=config.search_block_size)
    dense_rankings = {}
    def rows():
        for (qid, row), chunk_hits in zip(query_items, hits):
            dense_docs = aggregate_dense_docs(chunk_hits)
            doc_ids = [doc['doc_id'] for doc in dense_docs]
            dense_rankings[str(qid)] = doc_ids
            yield {'query_id': str(qid), 'question': row.get('question', '') if isinstance(row, dict) else '', 'gold': row.get('answer') if isinstance(row, dict) else None, 'dense_top_docs': dense_docs}
    append_jsonl(output_dense_file, rows())
    return dense_rankings

def write_fused_rankings(payload: dict[str, Any], step4_rankings: dict[str, list[str]], dense_rankings: dict[str, list[str]], output_fused_file: Path, fusion: dict[str, Any]) -> dict[str, list[str]]:
    fused_rankings = {}
    def rows():
        for qid, row in payload.items():
            qid = str(qid)
            fused = rrf_fuse(
                [(step4_rankings.get(qid, []), fusion['step4_weight']), (dense_rankings.get(qid, []), fusion['ft_dense_weight'])],
                rrf_k=fusion['rrf_k'],
                top_docs=100,
            )
            fused_rankings[qid] = fused
            yield {'query_id': qid, 'question': row.get('question', '') if isinstance(row, dict) else '', 'gold': row.get('answer') if isinstance(row, dict) else None, 'step4_doc_ids': step4_rankings.get(qid, []), 'ft_dense_doc_ids': dense_rankings.get(qid, []), 'fused_doc_ids': fused}
    append_jsonl(output_fused_file, rows())
    return fused_rankings

chunk_embeddings = encode_chunks_to_memmap(model, chunks, OUTPUT_DIR / 'embeddings' / 'chunk_embeddings_fp16.npy', batch_size=config.encode_batch_size)
print('chunk_embeddings:', chunk_embeddings.shape, chunk_embeddings.dtype)


## Tune Fusion On Dev And Create Public Submission

In [ ]:
step4_dev_rankings = load_step4_rankings(STEP4_DEV_RANKINGS)
dev_dense_rankings = dense_rank_payload(dev_payload, OUTPUT_DIR / 'rankings' / 'dev_rankings_ft_dense.jsonl')

trials = []
for rrf_k in config.rrf_k_values:
    for step4_weight in config.step4_weights:
        for ft_dense_weight in config.ft_dense_weights:
            fused = {}
            for qid in dev_payload:
                qid = str(qid)
                fused[qid] = rrf_fuse(
                    [(step4_dev_rankings.get(qid, []), step4_weight), (dev_dense_rankings.get(qid, []), ft_dense_weight)],
                    rrf_k=rrf_k,
                    top_docs=100,
                )
            metrics = evaluate_rankings(fused, dev_payload)['macro']
            trials.append({'rrf_k': rrf_k, 'step4_weight': step4_weight, 'ft_dense_weight': ft_dense_weight, 'metrics': metrics})
trials.sort(key=lambda row: (row['metrics']['recall@5'], row['metrics']['precision@5'], row['metrics']['recall@20'], row['metrics']['mrr']), reverse=True)
best_fusion = {k: trials[0][k] for k in ['rrf_k', 'step4_weight', 'ft_dense_weight']}
write_json(OUTPUT_DIR / 'metrics' / 'fusion_trials.json', trials)
write_json(OUTPUT_DIR / 'configs' / 'best_fusion_config.json', best_fusion)
print('best_fusion:', best_fusion)
print(json.dumps(trials[0]['metrics'], ensure_ascii=False, indent=2))

dev_fused_rankings = write_fused_rankings(
    dev_payload,
    step4_dev_rankings,
    dev_dense_rankings,
    OUTPUT_DIR / 'rankings' / 'dev_rankings_step5_fused.jsonl',
    best_fusion,
)
dev_metrics = evaluate_rankings(dev_fused_rankings, dev_payload)
write_json(OUTPUT_DIR / 'metrics' / 'dev_metrics_step5_fused.json', dev_metrics)
write_json(OUTPUT_DIR / 'predictions' / 'dev_predictions_top5_step5.json', {qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in dev_fused_rankings.items()})

# Export train candidates with the same Step 5 fusion contract, so Step 6 can
# mine semi-hard negatives from Step 5 candidates without any fallback.
step4_train_rankings = load_step4_rankings(STEP4_TRAIN_RANKINGS)
train_dense_rankings = dense_rank_payload(train_payload, OUTPUT_DIR / 'rankings' / 'train_rankings_ft_dense.jsonl')
train_fused_rankings = write_fused_rankings(
    train_payload,
    step4_train_rankings,
    train_dense_rankings,
    OUTPUT_DIR / 'rankings' / 'train_rankings_step5_fused.jsonl',
    best_fusion,
)
train_metrics = evaluate_rankings(train_fused_rankings, train_payload)
write_json(OUTPUT_DIR / 'metrics' / 'train_metrics_step5_fused.json', train_metrics)
write_json(OUTPUT_DIR / 'predictions' / 'train_predictions_top5_step5.json', {qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in train_fused_rankings.items()})

public_payload = read_json(PUBLIC_FILE)
step4_public_rankings = load_step4_rankings(STEP4_PUBLIC_RANKINGS)
public_dense_rankings = dense_rank_payload(public_payload, OUTPUT_DIR / 'rankings' / 'public_rankings_ft_dense.jsonl')
public_fused_rankings = write_fused_rankings(
    public_payload,
    step4_public_rankings,
    public_dense_rankings,
    OUTPUT_DIR / 'rankings' / 'public_rankings_step5_fused.jsonl',
    best_fusion,
)
public_predictions = {qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in public_fused_rankings.items()}
submission = make_submission(public_predictions)
validation = validate_submission_payload(submission, public_payload, valid_doc_ids)
submission_dir = OUTPUT_DIR / 'submission'
write_json(submission_dir / 'submission.json', submission)
write_json(submission_dir / 'submission_validation.json', validation)
if validation['num_errors']:
    raise ValueError(f"Submission validation failed: {validation['num_errors']} errors")
write_submission_zip(submission_dir / 'submission.json', submission_dir / 'submission.zip')

report = {
    'inputs': {
        'chunks_file': str(CHUNKS_FILE),
        'train_split_file': str(TRAIN_FILE),
        'dev_split_file': str(DEV_FILE),
        'step4_best_config': str(STEP4_BEST_CONFIG) if STEP4_BEST_CONFIG else None,
        'step4_config_file': str(STEP4_CONFIG_FILE) if STEP4_CONFIG_FILE else None,
        'step4_train_rankings': str(STEP4_TRAIN_RANKINGS),
        'step4_dev_rankings': str(STEP4_DEV_RANKINGS),
        'step4_public_rankings': str(STEP4_PUBLIC_RANKINGS),
    },
    'config': {**asdict(config), 'active_max_seq_length': model.max_seq_length, 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES')},
    'best_fusion': best_fusion,
    'dev_macro': dev_metrics['macro'],
    'train_macro': train_metrics['macro'],
    'model_output_path': str(model_output_path),
    'public_outputs': {
        'rankings': 'rankings/public_rankings_step5_fused.jsonl',
        'submission_zip': 'submission/submission.zip',
        'submission_validation': 'submission/submission_validation.json',
    },
    'next_step6_inputs': [
        'chunks.jsonl from existing step4.zip dataset',
        'train_split.json from existing step4.zip dataset',
        'dev_split.json from existing step4.zip dataset',
        'step5/rankings/train_rankings_step5_fused.jsonl',
        'step5/rankings/dev_rankings_step5_fused.jsonl',
        'step5/rankings/public_rankings_step5_fused.jsonl',
        'step5/models/vietnamese-bi-encoder-mnrl if reusing the fine-tuned retriever',
    ],
}
write_json(OUTPUT_DIR / 'reports' / 'run_report.json', report)
print(json.dumps(dev_metrics['macro'], ensure_ascii=False, indent=2))
print('Submission:', submission_dir / 'submission.zip')


## Files To Download

In [ ]:
for path in [
    OUTPUT_DIR / 'submission' / 'submission.zip',
    OUTPUT_DIR / 'submission' / 'submission_validation.json',
    OUTPUT_DIR / 'metrics' / 'train_metrics_step5_fused.json',
    OUTPUT_DIR / 'metrics' / 'dev_metrics_step5_fused.json',
    OUTPUT_DIR / 'metrics' / 'fusion_trials.json',
    OUTPUT_DIR / 'rankings' / 'train_rankings_step5_fused.jsonl',
    OUTPUT_DIR / 'rankings' / 'dev_rankings_step5_fused.jsonl',
    OUTPUT_DIR / 'rankings' / 'public_rankings_step5_fused.jsonl',
    OUTPUT_DIR / 'reports' / 'run_report.json',
    OUTPUT_DIR / 'reports' / 'model_manifest.json',
    OUTPUT_DIR / 'models' / 'vietnamese-bi-encoder-mnrl',
]:
    print(path, 'OK' if path.exists() else 'MISSING')
